##### Imports and setup

In [1]:
# setup the notebook environment
!pixi install
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import logging
logging.getLogger("pypsa.network.descriptors").setLevel(logging.ERROR)

def ensure_project_root_on_path(marker_dir: str = "modules") -> str:
    """Find the nearest ancestor folder containing `marker_dir`, add it to
    sys.path (front) and return its path. Raises RuntimeError if not found.
    """
    cwd = Path.cwd().resolve()
    # If cwd contains marker_dir, use cwd
    if (cwd / marker_dir).exists():
        p = str(cwd)
        if p not in sys.path:
            sys.path.insert(0, p)
        return p
    # Walk parents
    for parent in cwd.parents:
        if (parent / marker_dir).exists():
            p = str(parent)
            if p not in sys.path:
                sys.path.insert(0, p)
            return p
    raise RuntimeError(f"Could not find '{marker_dir}' directory in {cwd} or any parent")


project_root = ensure_project_root_on_path()
print("Added project root to sys.path:", project_root)

# imports and loading data
from modules.analysis_toolkit.analyzer import ResultsComputer, GeoOptions, IndexFinder
from modules.analysis_toolkit.helpers.plotting import TimeSeriesPlot, HistogramPlot, BarChartPlot, WaterfallPlot
import pandas as pd

study_years = [2030, 2040]
rc = {year: ResultsComputer(year=year, apply_snapshot_filter=False) for year in study_years}
weights = rc[2030].ns.get_iem_dispatch().snapshot_weightings["generators"]

✔ The default environment has been installed.
Added project root to sys.path: /Users/rca/PycharmProjects/NGV-FBMC


INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Status Quo (SQ) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Integrated Energy Market (IEM) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Flow-based market coupling (FBMC) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'SQ (2030) - redispatch' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, sto

## Average DAM consumer prices in GB

In [2]:
pd.concat(
    [
        rc[year].consumer_costs_in_gb.compare_dispatch() \
            .groupby("bus", axis=0).sum() \
            .mul(weights, level="snapshot", axis=1) \
            .groupby("scenario", axis=1).sum() \
            .abs() \
            .sum() \
            .div(
                rc[year].withdrawal.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
                    .drop("Line", level="component") \
                    .drop(["DC", "DC_OH"], level="carrier") \
                    .query("bus.str.contains('GB ')") \
                    .groupby("bus", axis=0).sum() \
                    .groupby("scenario", axis=1).sum() \
                    .sum(),
                fill_value=0
            ) \
            .round(2) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
).loc[["iem", "iem_fb", "sq"]]

Calling method consumer_costs_in_gb


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23431/1273281252.py:4: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23431/1273281252.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method withdrawal


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23431/1273281252.py:14: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23431/1273281252.py:15: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method consumer_costs_in_gb


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23431/1273281252.py:4: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23431/1273281252.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method withdrawal


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23431/1273281252.py:14: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23431/1273281252.py:15: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
iem,43.08,18.02
iem_fb,43.15,18.26
sq,42.95,17.98


# Consumer costs

In [3]:
# note: run each of the components first
total_consumer_costs = dispatch_consumer_cost + cfd_compensation + redispatch_costs
total_consumer_costs = total_consumer_costs.loc[["sq", "iem", "iem_fb"]]
total_consumer_costs.loc["iem_fb"] = total_consumer_costs.loc["iem_fb"] + compensation_ic_constraints

total_consumer_costs.loc["sq"] = total_consumer_costs.loc["sq"]

total_consumer_costs.loc["iem-sq"] = total_consumer_costs.loc["iem"] - total_consumer_costs.loc["sq"]
total_consumer_costs.loc["iem_fb-iem"] = total_consumer_costs.loc["iem_fb"] - total_consumer_costs.loc["iem"]
total_consumer_costs

NameError: name 'dispatch_consumer_cost' is not defined

## Dispatch consumer costs

In [ ]:
dispatch_consumer_cost = -pd.concat(
    [
        rc[year].consumer_costs_in_gb.compare_dispatch() \
            .mul(weights, level="snapshot", axis=1) \
            .groupby("scenario", axis=1).sum() \
            .sum()
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

dispatch_consumer_cost

## CfD compensation

In [ ]:
from itertools import product
year_scenario_combinations = list(product(study_years, ["sq", "iem", "iem_fb"]))

cfd_compensation = pd.concat(
    [
        rc[year].cfd_compensation_costs(scenario) \
            .mul(weights, level="snapshot", axis=1) \
            .sum() \
            .agg({"carrier": "sum"}) \
            .div(1e6) \
            .round(1) \
        for year, scenario in year_scenario_combinations
    ],
    keys=year_scenario_combinations,
    names=["year", "scenario"],
    axis=0
).droplevel(2).unstack("year")

cfd_compensation.loc["diff: iem-sq"] = cfd_compensation.loc["iem"] - cfd_compensation.loc["sq"]
cfd_compensation.loc["diff: iemfb-iem"] = cfd_compensation.loc["iem_fb"] - cfd_compensation.loc["iem"]

cfd_compensation

### Renewable curtailment volumes (ramp up/ramp down)
To try to explain the reduction in cfd_compensation costs

## Redispatch costs

In [ ]:
redispatch_costs = pd.concat(
    [
        rc[year].constraint_costs.compare_redispatch() 
            .mul(weights, level="snapshot", axis=1) \
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

redispatch_costs

## Compensation to merchant interconnectors (eventual)

In [ ]:
compensation_ic_constraints = pd.concat(
    [
        rc[year].lost_congestion_income() \
            .mul(weights, level="snapshot", axis=1) \
            .sum(axis=1) \
            .agg({"name": "sum"}) \
            .div(1e6) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
).reset_index(drop=True).T.squeeze()

compensation_ic_constraints

### breakdown per interconnector
In 2030, BritNed, IFA and IFA2 and Eleclink lose congestion income due to flow-based constraints, adding up to 4.4 M€.
In 2040, some interconnectors lose but others gain congestion income, with a net loss of 12.8 M€.

In [ ]:
pd.concat(
    [
        rc[year].lost_congestion_income() \
            .mul(weights, level="snapshot", axis=1) \
            .sum(axis=1)
            .div(1e6) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

# Merchant CI

In [ ]:
# note: run each of the components first
pd.concat({
    "SQ": captured_sq,
    "IEM": captured_sq + delta_price_coupling + delta_capture_rate,
    "IEM-FB": captured_sq + delta_price_coupling + delta_capture_rate + delta_gb_constraints,
})
# assuming no compensation for IEM-FB

## Captured SQ CI

In [ ]:
captured_sq = pd.concat(
    [
        rc[year].congestion_income.sq_dispatch(where=GeoOptions.GB_ONLY, apply_capture_rates=True) \
            .mul(weights, level="snapshot", axis=1) \
            .sum() \
            .agg({"name": "sum"}) \
            .div(1e6) \
            .round(1)  
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
captured_sq

## ∆ price coupling (price and volume effects)

In [ ]:
delta_price_coupling = pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .mul(weights, level="snapshot", axis=1) \
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
).loc["diff: iem-sq"]

delta_price_coupling

## ∆ capture rate

In [ ]:
delta_capture_rate = pd.concat(
    [
        rc[year].congestion_income.sq_dispatch(where=GeoOptions.GB_ONLY, apply_capture_rates=False) \
            .mul(weights, level="snapshot", axis=1) \
            .sum(axis=1) \
            .agg({"name": "sum"}) \
            .div(1e6) \
            .round(1).sub(
                rc[year].congestion_income.sq_dispatch(where=GeoOptions.GB_ONLY, apply_capture_rates=True) \
                    .mul(weights, level="snapshot", axis=1) \
                    .sum(axis=1) \
                    .agg({"name": "sum"}) \
                    .div(1e6) \
                    .round(1)
            )
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

delta_capture_rate

## ∆ GB constraints (FB reflecting GB grid in dispatch)

In [ ]:
delta_gb_constraints = pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .mul(weights, level="snapshot", axis=1) \
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
).loc["diff: iemfb-iem"]

delta_gb_constraints

## ∆ compensation flow limits

In [ ]:
delta_compensation = pd.concat(
    [
        rc[year].lost_congestion_income() \
            .mul(weights, level="snapshot", axis=1) \
            .sum(axis=1) \
            .agg({"name": "sum"}) \
            .div(1e6) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

delta_compensation